In [1]:
from sklearn.linear_model import LassoCV
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np
import seaborn as sns

In [25]:
live_avg = pd.read_csv("../data/samples/live_avg.csv")
live_avg_norm = pd.read_csv("../data/samples/live_avg_norm.csv")
live_avg_scaled = pd.read_csv("../data/samples/live_avg_scaled.csv")
weighted_metrics = pd.read_csv("../data/samples/weighted_metrics_data.csv")

In [3]:
to_keep = ['GPU Utilization (%)',
 'Power Draw (Watts)',
 'GPU Current Clock (MHz)',
 'Memory Allocation Used (MB)',
 'Current Time']
to_drop = ['Memory Utilization (%)', 'Time Delta', 'Iteration', 'GPU Clock Utilization', 'GPU Temp (°C)',]

In [4]:
y = live_avg.iloc[:, -2:-1]
y = y['Resonse Time'].tolist()
live_avg = live_avg.iloc[:, :-1]

In [5]:
live_avg_norm = live_avg_norm.drop(columns=['Unnamed: 0'])
live_avg = live_avg.drop(columns=['Unnamed: 0'])
live_avg_scaled = live_avg_scaled.drop(columns=['Unnamed: 0'])

In [6]:
live_avg_norm.columns = live_avg.columns[:-1]
live_avg_norm_aug = live_avg_norm.drop(columns=to_drop)
live_avg_norm = live_avg_norm.drop(columns=['Iteration'])

In [26]:
weighted_metrics

,GPU Utilization (%),Power Draw (Watts),GPU Current Clock (MHz),Memory Allocation Used (MB),Response Time,GPU Utilization (%) Weight,Power Draw (Watts) Weight,GPU Current Clock (MHz) Weight,Memory Allocation Used (MB) Weight,Linear Combination
0,0.000000e+00,1.074727e-17,6.111549e-11,4.272282e-12,2.606603,8.320419e+18,8.797574e+17,-1.071177e+11,-3.912974e+08,2.906767
1,0.000000e+00,1.074727e-17,6.111549e-11,4.272282e-12,0.522210,8.320419e+18,8.797574e+17,-1.071177e+11,-3.912974e+08,2.906767
2,0.000000e+00,1.074727e-17,6.111549e-11,4.272282e-12,2.582972,8.320419e+18,8.797574e+17,-1.071177e+11,-3.912974e+08,2.906767
3,9.054146e-21,4.608334e-17,3.429258e-10,2.013194e-09,0.955865,8.320419e+18,8.797574e+17,-1.071177e+11,-3.912974e+08,3.096299
4,1.810829e-20,8.141941e-17,6.247361e-10,3.017655e-09,1.145166,8.320419e+18,8.797574e+17,-1.071177e+11,-3.912974e+08,3.678874
...,...,...,...,...,...,...,...,...,...,...
64,2.723393e-19,8.271894e-17,6.247362e-10,3.019555e-09,13.239929,8.320419e+18,8.797574e+17,-1.071177e+11,-3.912974e+08,6.936692
65,3.357840e-19,8.009503e-17,6.247362e-10,3.019555e-09,6.724487,8.320419e+18,8.797574e+17,-1.071177e+11,-3.912974e+08,5.156173
66,3.666930e-19,7.976252e-17,6.247362e-10,3.019555e-09,9.312371,8.320419e+18,8.797574e+17,-1.071177e+11,-3.912974e+08,5.120824
67,3.636750e-19,8.490075e-17,6.247362e-10,3.019555e-09,7.641057,8.320419e+18,8.797574e+17,-1.071177e+11,-3.912974e+08,9.616108


In [7]:
X_train, X_test, y_train, y_test = train_test_split(live_avg_norm_aug, y, test_size=0.2, shuffle=True, random_state=20)

In [18]:
lcv = LassoCV(cv=5)
lcv.fit(X_train, y_train)
score = lcv.score(X_test, y_test)
print(f"Accuracy: {score}")
print(f"Weights: {lcv.coef_} & Alpha: {lcv.alpha_}")

Accuracy: 0.1526489889790924
Weights: [0.00000000e+00 0.00000000e+00 0.00000000e+00 2.35544021e+09
 0.00000000e+00] & Alpha: 1.1536258593315319e-10


c:\Users\rahul\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:683: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 5.795806578831616, tolerance: 1.486216030391058
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\rahul\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:683: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 5.485279394701138, tolerance: 1.486216030391058
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\rahul\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:683: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 87.33190913724684, tolerance: 1.486216030391058
  model = cd_fast.enet_coordinate_descent_gram(


In [9]:
len(X_test)

14

In [10]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import Lasso, Ridge

rfe = RFE(estimator=Lasso(), n_features_to_select=2, verbose=True)

In [11]:
rfe.fit(live_avg_norm.iloc[:55, :], y_train)
rfe.score(live_avg_norm.iloc[55:, :], y_test)

Fitting estimator with 9 features.
Fitting estimator with 8 features.
Fitting estimator with 7 features.
Fitting estimator with 6 features.
Fitting estimator with 5 features.
Fitting estimator with 4 features.
Fitting estimator with 3 features.


-0.09005972634424753

In [14]:
dict(zip(live_avg_norm.columns.tolist(), rfe.support_))

{'GPU Utilization (%)': False,
 'Power Draw (Watts)': False,
 'GPU Temp (°C)': False,
 'GPU Current Clock (MHz)': False,
 'Memory Allocation Used (MB)': False,
 'Memory Utilization (%)': False,
 'Current Time': False,
 'Time Delta': True,
 'GPU Clock Utilization': True}

In [24]:
# live_avg_norm_aug = live_avg_norm_aug.drop(columns=['Current Time'])
live_avg_norm_aug['Response Time'] = y
live_avg_norm_aug.to_csv('../data/samples/live_avg_norm_aug.csv')